# Phase 3: Build a Strong Retrieval Pipeline

## Step 11: Query Transformation

### Learning

- Standalone query rewriting
- Context-dependent questions
- Query expansion
- Multi-query retrieval
- Query decomposition
- Retrieval routing
- Hypothetical document embeddings
- Query intent classification

---

## Key Takeaways

- The user's literal wording isn't always what should be searched for. "When did
  he do it?" means nothing to a retriever without the conversation before it —
  retrieval needs a **standalone query**, but generation should still answer the
  **original message**.
- Rewriting can quietly change what's being asked (dropping a negation or a scope
  word like "regular" vs "extended"). That's a correctness risk, not just style.
- Asking the same question multiple ways (expansion) or breaking it into parts
  (decomposition) both raise the odds the right chunk gets retrieved.
- Not every message should hit the retriever — a greeting, an identifier lookup,
  and an open question all deserve different handling. That's what a router is for.

---

## To do (mirrors the Roadmap 1:1)

1. Detect context-dependent queries
2. Rewrite follow-ups into standalone queries
3. Preserve user intent
4. Generate multiple query variations
5. Fuse multi-query results
6. Detect multi-part questions
7. Combine evidence across sub-questions
8. Build a retrieval router
9. Experiment with hypothetical-document retrieval (HyDE)
10. Log every transformation

Each section below is kept intentionally simple and self-contained — a little
repeated code across sections is fine; it's easier to follow than one clever
shared function doing several jobs.

## 0. Environment Setup

Same connection pattern as every step since Step 4. This notebook indexes its
own small copy of the fictional **ByteMage** corpus (same company/persona as
Steps 7, 9, and 10) so it runs standalone.

In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

try:
    response = es.info()
    print(response)
except Exception as e:
    print("TYPE:", type(e).__name__)
    print("ERROR:")
    print(e)
    if hasattr(e, "body"):
        print("\nBODY:")
        print(e.body)

import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config
importlib.reload(config)

{'name': '0e1f447b24d9', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'lHCCr2OyQ0yMLKjGmwdmBg', 'version': {'number': '8.19.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '93788a8c2882eb5b606510680fac214cff1c7a22', 'build_date': '2025-07-23T22:10:18.138212839Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [2]:
import re
from typing import Literal

import chromadb
from openai import OpenAI
from pydantic import BaseModel, Field

from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma = chromadb.HttpClient(host="localhost", port=8000)

INDEX_NAME = "rag_documents_query_transform"
COLLECTION_NAME = "bytemage_query_transform_docs"

In [3]:
bytemage_documents = [
    {
        "chunk_id": "company-overview-001",
        "document_id": "company-overview",
        "title": "ByteMage Company Overview",
        "text": (
            "ByteMage was founded in 2015 by Alan Whitfield and Priya Kapoor. "
            "The company is headquartered in Austin, Texas, and builds cloud "
            "applications, data platforms, and AI-powered business tools."
        ),
    },
    {
        "chunk_id": "leave-policy-001",
        "document_id": "leave-policy",
        "title": "ByteMage Leave Policy",
        "text": (
            "ByteMage employees may take up to five sick days per month without "
            "additional approval. Extended sick leave beyond five days requires "
            "notifying HR within 48 hours and is approved by the employee's "
            "direct manager. Unused sick days do not roll over to the next month "
            "and are forfeited at month end."
        ),
    },
    {
        "chunk_id": "compensation-policy-001",
        "document_id": "compensation-policy",
        "title": "ByteMage Compensation Policy",
        "text": (
            "ByteMage salary bands are reviewed every March. Senior Software "
            "Engineers in the AI Platform department fall in Band E5."
        ),
    },
    {
        "chunk_id": "data-retention-policy-001",
        "document_id": "data-retention-policy",
        "title": "ByteMage Data Retention Policy",
        "text": (
            "ByteMage retains customer support data for a minimum of seven "
            "years under regulatory requirement RX-118. Data may be deleted "
            "earlier only upon a verified customer request."
        ),
    },
    {
        "chunk_id": "engineering-handbook-001",
        "document_id": "engineering-handbook",
        "title": "ByteMage Engineering Handbook",
        "text": (
            "ByteMage pull requests require at least one approving review from "
            "a senior engineer before merging to main. John Doe co-authored "
            "this standard as part of the AI Platform team's review guidelines."
        ),
    },
    {
        "chunk_id": "product-roadmap-001",
        "document_id": "product-roadmap",
        "title": "ByteMage Product Roadmap",
        "text": (
            "As of Q3 2026, ByteMage's top product priority is launching the "
            "AI Search Platform's new billing dashboard for enterprise "
            "customers."
        ),
    },
    {
        "chunk_id": "onboarding-guide-001",
        "document_id": "onboarding-guide",
        "title": "ByteMage Onboarding Guide",
        "text": (
            "New ByteMage employees complete orientation during their first "
            "week, including IT setup, benefits enrollment, and an "
            "introduction to the AI Search Platform."
        ),
    },
]

print(f"Loaded {len(bytemage_documents)} chunks.")

Loaded 7 chunks.


In [4]:
# ---- Elasticsearch (lexical side) ----
from elasticsearch.helpers import bulk

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "document_id": {"type": "keyword"},
            "title": {"type": "text"},
            "text": {"type": "text"},
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)

actions = [
    {"_index": INDEX_NAME, "_id": chunk["chunk_id"], "_source": chunk}
    for chunk in bytemage_documents
]
success, failed = bulk(es, actions)
print("Successfully indexed into Elasticsearch:", success, "| Failed:", failed)

Successfully indexed into Elasticsearch: 7 | Failed: []


In [5]:
# ---- Chroma (semantic side) ----
try:
    client_chroma.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client_chroma.get_or_create_collection(name=COLLECTION_NAME)

ids, texts, embeddings, metadatas = [], [], [], []

for chunk in bytemage_documents:
    embedding_response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunk["text"])
    ids.append(chunk["chunk_id"])
    texts.append(chunk["text"])
    embeddings.append(embedding_response.data[0].embedding)
    metadatas.append({"document_id": chunk["document_id"], "title": chunk["title"]})

collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)
print(f"Indexed {collection.count()} documents into Chroma.")

Indexed 7 documents into Chroma.


### Basic retrieval

Plain vector search, lexical search, and hybrid search (RRF over the two) —
exactly the functions built in Step 7, copied here so this notebook doesn't
depend on another one. Everything below builds on top of `hybrid_search`.

In [6]:
def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def vector_search(query_text, top_k=5):
    query_embedding = get_embedding(query_text)
    raw = collection.query(query_embeddings=[query_embedding], n_results=top_k)

    results = []
    for i, chunk_id in enumerate(raw["ids"][0]):
        results.append({
            "chunk_id": chunk_id,
            "text": raw["documents"][0][i],
            "rank": i + 1,
        })
    return results


def lexical_search(query_text, top_k=5):
    raw = es.search(index=INDEX_NAME, body={
        "size": top_k,
        "query": {"match": {"text": query_text}},
    })

    results = []
    for i, hit in enumerate(raw["hits"]["hits"]):
        results.append({
            "chunk_id": hit["_source"]["chunk_id"],
            "text": hit["_source"]["text"],
            "rank": i + 1,
        })
    return results


def hybrid_search(query_text, top_k=5):
    """Same idea as Step 7: fuse vector + lexical rankings with RRF (k=60)."""
    vector_results = vector_search(query_text, top_k=10)
    lexical_results = lexical_search(query_text, top_k=10)

    scores = {}
    chunks = {}

    for result in vector_results:
        chunk_id = result["chunk_id"]
        chunks[chunk_id] = result["text"]
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (60 + result["rank"])

    for result in lexical_results:
        chunk_id = result["chunk_id"]
        chunks[chunk_id] = result["text"]
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (60 + result["rank"])

    ranked_chunk_ids = sorted(scores, key=scores.get, reverse=True)

    results = []
    for chunk_id in ranked_chunk_ids[:top_k]:
        results.append({"chunk_id": chunk_id, "text": chunks[chunk_id], "score": scores[chunk_id]})
    return results

In [7]:
for r in hybrid_search("Who founded ByteMage?"):
    print(r["chunk_id"], round(r["score"], 4), "-", r["text"][:70])

onboarding-guide-001 0.0323 - New ByteMage employees complete orientation during their first week, i
company-overview-001 0.0323 - ByteMage was founded in 2015 by Alan Whitfield and Priya Kapoor. The c
compensation-policy-001 0.0313 - ByteMage salary bands are reviewed every March. Senior Software Engine
data-retention-policy-001 0.0312 - ByteMage retains customer support data for a minimum of seven years un
engineering-handbook-001 0.0308 - ByteMage pull requests require at least one approving review from a se


## 1. Detect Context-Dependent Queries

> `User: Who founded the company? / Assistant: Alan founded the company. /
> User: When did he do it?` — the final message can't be searched effectively
> without conversation context.

Simplest possible check: does the message contain a pronoun that could be
referring to something said earlier? It's not perfect, but it's cheap and easy
to reason about — good enough to decide "should I bother rewriting this?"

In [8]:
def looks_context_dependent(message):
    pronouns = ["he", "she", "it", "they", "him", "her", "them", "this", "that", "these", "those"]
    words = message.lower().replace("?", "").split()
    return any(pronoun in words for pronoun in pronouns)


print(looks_context_dependent("When did he do it?"))
print(looks_context_dependent("How many sick days do employees get?"))
print(looks_context_dependent("Does that apply to extended leave too?"))

True
False
True


## 2. Rewrite Follow-Ups into Standalone Queries

> Send the conversation history and current message to a model. Require
> structured output: `{"standalone_query": ..., "needs_retrieval": true}`. Use
> the rewritten query only for retrieval — the original message still goes to
> the final answer-generation step.

In [9]:
class QueryRewrite(BaseModel):
    standalone_query: str
    needs_retrieval: bool


def rewrite_query(conversation_history, latest_message):
    history_text = "\n".join(f"{turn['role']}: {turn['content']}" for turn in conversation_history)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Rewrite the user's latest message into a standalone search query, "
                    "using the conversation to resolve pronouns like 'he' or 'it'. Also "
                    "say whether answering this message requires searching documents at all."
                ),
            },
            {"role": "user", "content": f"Conversation:\n{history_text}\n\nLatest message: {latest_message}"},
        ],
        response_format=QueryRewrite,
    )
    return response.choices[0].message.parsed

In [10]:
conversation_history = [
    {"role": "user", "content": "Who founded ByteMage?"},
    {"role": "assistant", "content": "ByteMage was founded by Alan Whitfield and Priya Kapoor."},
]
latest_message = "When did he do it?"

rewrite = rewrite_query(conversation_history, latest_message)

print("Original:  ", latest_message)
print("Standalone:", rewrite.standalone_query)
print("Needs retrieval:", rewrite.needs_retrieval)

Original:   When did he do it?
Standalone: When did Alan Whitfield found ByteMage?
Needs retrieval: True


## 3. Preserve User Intent

> The rewritten query must preserve names, dates, negations, and requested
> scope — and must never try to answer the question itself.

The prompt above is too loose: nothing stops it from dropping a scope word.
Example: the follow-up below narrows the topic from "extended sick leave" to
specifically "regular" sick days. A careless rewrite loses that distinction.

In [11]:
REWRITE_RULES = """Rewrite the user's latest message into a standalone search query, \
using the conversation to resolve references like 'he' or 'it'.

Rules:
- Keep all names, dates, and identifiers exactly as given.
- Keep any negation ("not", "does not", "excluding") from the conversation.
- Keep the requested scope. If the user narrows the topic (e.g. from "extended \
  leave" to "regular leave"), the rewrite must reflect the NEW, narrower scope.
- Do NOT answer the question — output a search query, not a fact.
- Do NOT add details that were never mentioned.

Also say whether answering this message requires searching documents at all.
"""


def rewrite_query(conversation_history, latest_message):
    history_text = "\n".join(f"{turn['role']}: {turn['content']}" for turn in conversation_history)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": REWRITE_RULES},
            {"role": "user", "content": f"Conversation:\n{history_text}\n\nLatest message: {latest_message}"},
        ],
        response_format=QueryRewrite,
    )
    return response.choices[0].message.parsed

In [12]:
scope_conversation = [
    {"role": "user", "content": "Does ByteMage allow skipping the notification for extended sick leave?"},
    {"role": "assistant", "content": "No — extended sick leave requires notifying HR within 48 hours."},
]
followup = "What about for regular sick days, not extended ones?"

rewrite = rewrite_query(scope_conversation, followup)

print("Original:  ", followup)
print("Standalone:", rewrite.standalone_query)

Original:   What about for regular sick days, not extended ones?
Standalone: Does ByteMage allow skipping the notification for regular sick days, excluding extended sick leave?


## 4. Generate Multiple Query Variations

> Ask the model for three search variations: `{"queries": [...]}`. Each
> variation should use different wording for the same intent.

In [13]:
class QueryVariations(BaseModel):
    queries: list[str]


def generate_query_variations(standalone_query):
    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Write exactly three alternative search queries with the same "
                    "meaning as the input, using different words or phrasing."
                ),
            },
            {"role": "user", "content": standalone_query},
        ],
        response_format=QueryVariations,
    )
    return response.choices[0].message.parsed.queries


standalone_query = "When did Alan found ByteMage?"
variations = generate_query_variations(standalone_query)

print("Standalone query:", standalone_query)
for q in variations:
    print("-", q)

Standalone query: When did Alan found ByteMage?
- In what year did Alan establish ByteMage?
- When was ByteMage founded by Alan?
- What is the founding date of ByteMage by Alan?


## 5. Fuse Multi-Query Results

> Merge all retrieved chunks by stable ID. Use rank fusion or a simple
> weighted method. Record which query variation retrieved each chunk.

Simplest useful rule: run `hybrid_search` for every variation, then a chunk
that shows up for **more** variations is probably more reliably relevant than
one that only showed up for one phrasing. We just count how many variations
found each chunk, and keep track of which ones.

In [14]:
def multi_query_search(queries, top_k=5):
    found_by = {}   # chunk_id -> list of queries that retrieved it
    chunk_text = {}

    for query in queries:
        results = hybrid_search(query, top_k=5)
        for result in results:
            chunk_id = result["chunk_id"]
            chunk_text[chunk_id] = result["text"]
            found_by.setdefault(chunk_id, []).append(query)

    # More variations found it -> ranked higher.
    ranked_chunk_ids = sorted(found_by, key=lambda chunk_id: len(found_by[chunk_id]), reverse=True)

    results = []
    for chunk_id in ranked_chunk_ids[:top_k]:
        results.append({
            "chunk_id": chunk_id,
            "text": chunk_text[chunk_id],
            "found_by": found_by[chunk_id],
        })
    return results


fused_results = multi_query_search(variations)

for r in fused_results:
    print(r["chunk_id"], "| found by", len(r["found_by"]), "variation(s):", r["found_by"])

company-overview-001 | found by 3 variation(s): ['In what year did Alan establish ByteMage?', 'When was ByteMage founded by Alan?', 'What is the founding date of ByteMage by Alan?']
onboarding-guide-001 | found by 3 variation(s): ['In what year did Alan establish ByteMage?', 'When was ByteMage founded by Alan?', 'What is the founding date of ByteMage by Alan?']
engineering-handbook-001 | found by 3 variation(s): ['In what year did Alan establish ByteMage?', 'When was ByteMage founded by Alan?', 'What is the founding date of ByteMage by Alan?']
compensation-policy-001 | found by 2 variation(s): ['In what year did Alan establish ByteMage?', 'When was ByteMage founded by Alan?']
data-retention-policy-001 | found by 2 variation(s): ['In what year did Alan establish ByteMage?', 'What is the founding date of ByteMage by Alan?']


## 6. Detect Multi-Part Questions

> Test: "What is the leave allowance, who approves it, and what happens to
> unused leave?" Ask the model to decompose it: `{"subquestions": [...]}`.

In [15]:
class Decomposition(BaseModel):
    subquestions: list[str]


def decompose_question(question):
    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "If the question has multiple distinct parts, split it into separate "
                    "atomic sub-questions. If it's already a single question, return it "
                    "unchanged as the only item in the list."
                ),
            },
            {"role": "user", "content": question},
        ],
        response_format=Decomposition,
    )
    return response.choices[0].message.parsed.subquestions


multi_part_question = "What is the leave allowance, who approves it, and what happens to unused leave?"
subquestions = decompose_question(multi_part_question)

print("Question:", multi_part_question)
for subq in subquestions:
    print("-", subq)

print()
print("Single question:", decompose_question("What is ByteMage's sick leave allowance?"))

Question: What is the leave allowance, who approves it, and what happens to unused leave?
- What is the leave allowance?
- Who approves the leave allowance?
- What happens to unused leave?

Single question: ["What is ByteMage's policy on sick leave allowance?"]


## 7. Combine Evidence Across Sub-Questions

> Group retrieved chunks by sub-question. Pass the grouped evidence to the
> generation model. Require the final answer to address every sub-question.

In [16]:
def answer_multi_part_question(question):
    subquestions = decompose_question(question)

    context = ""
    for subq in subquestions:
        results = hybrid_search(subq, top_k=3)
        context += f"\nSub-question: {subq}\n"
        for r in results:
            context += f"- {r['text']}\n"

    system_prompt = (
        "Answer the user's full question using only the evidence below, which is "
        "grouped by sub-question. Your answer must address every sub-question.\n"
        + context
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


print(answer_multi_part_question(multi_part_question))

ByteMage employees are allowed up to five sick days per month without needing additional approval. If sick leave extends beyond five days in a month, the employee must notify HR within 48 hours, and the extended leave requires approval from the employee's direct manager. Any unused sick days at the end of the month do not roll over and are forfeited.


## 8. Build a Retrieval Router

> Classify each query into: No retrieval needed, Vector retrieval, Lexical
> retrieval, Hybrid retrieval, Database lookup, Clarification needed. Start
> with rule-based routing, then compare with model-based routing.

Simple rules first, since they're free and obviously correct when they match.
Anything the rules don't recognize falls back to asking the model.

In [17]:
def rule_based_route(query_text):
    text = query_text.strip().lower()

    greetings = ["hi", "hello", "hey", "thanks", "thank you"]
    if any(text.startswith(greeting) for greeting in greetings):
        return "no_retrieval"

    # Identifier-like strings, e.g. "RX-118"
    if re.search(r"[A-Z]{2,}-\d+", query_text):
        return "lexical"

    return None  # not sure -- let the model decide


class RouteDecision(BaseModel):
    route: Literal["no_retrieval", "vector", "lexical", "hybrid", "database_lookup", "clarification_needed"]


def model_based_route(query_text):
    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify the query into one route:\n"
                    "no_retrieval = small talk, no documents needed\n"
                    "lexical = exact identifiers/codes/names\n"
                    "vector = open-ended conceptual question\n"
                    "hybrid = conceptual question that also has specific terms\n"
                    "database_lookup = a structured fact a database would answer directly\n"
                    "clarification_needed = too vague to search"
                ),
            },
            {"role": "user", "content": query_text},
        ],
        response_format=RouteDecision,
    )
    return response.choices[0].message.parsed.route


def route_query(query_text):
    route = rule_based_route(query_text)
    if route is not None:
        return route, "rule"
    return model_based_route(query_text), "model"

In [18]:
test_queries = [
    "Hello!",
    "What does RX-118 require?",
    "How does ByteMage balance engineering speed with review quality?",
]

for q in test_queries:
    route, method = route_query(q)
    print(f"{route:<20} ({method}) <- {q!r}")

no_retrieval         (rule) <- 'Hello!'
lexical              (rule) <- 'What does RX-118 require?'
vector               (model) <- 'How does ByteMage balance engineering speed with review quality?'


## 9. Experiment with Hypothetical-Document Retrieval (HyDE)

> Ask a model to write a short hypothetical passage that would answer the
> question. Embed that passage instead of the raw question, and retrieve
> with it. Compare against direct query embedding.

A casually-phrased question and a formal policy sentence look very different
as text, even when they're about the same fact. Embedding a fake-but-
plausible *answer* (written in the corpus's style) can match the real
document better than embedding the raw, differently-styled question.

In [19]:
def generate_hypothetical_document(question):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Write a short 2-3 sentence passage, in the flat style of a company "
                    "policy document, that would answer the question. State it as fact. "
                    "This text is only used for search -- it is never shown to the user."
                ),
            },
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


def hyde_search(question, top_k=3):
    hypothetical_document = generate_hypothetical_document(question)
    embedding = get_embedding(hypothetical_document)

    raw = collection.query(query_embeddings=[embedding], n_results=top_k)
    results = [
        {"chunk_id": chunk_id, "text": raw["documents"][0][i]}
        for i, chunk_id in enumerate(raw["ids"][0])
    ]
    return results, hypothetical_document

In [20]:
casual_question = "gotta know -- does ByteMage support keep my info forever or what?"

direct_results = vector_search(casual_question, top_k=3)
hyde_results, hypothetical_document = hyde_search(casual_question, top_k=3)

print("Hypothetical document:", hypothetical_document)

print("\nDirect query embedding:")
for r in direct_results:
    print("-", r["chunk_id"], "-", r["text"][:70])

print("\nHyDE:")
for r in hyde_results:
    print("-", r["chunk_id"], "-", r["text"][:70])

Hypothetical document: ByteMage Support retains user information only as long as necessary to provide support services and comply with legal obligations. Personal data is securely deleted or anonymized after the retention period expires. Users can request data deletion in accordance with ByteMage's data privacy policies.

Direct query embedding:
- data-retention-policy-001 - ByteMage retains customer support data for a minimum of seven years un
- product-roadmap-001 - As of Q3 2026, ByteMage's top product priority is launching the AI Sea
- onboarding-guide-001 - New ByteMage employees complete orientation during their first week, i

HyDE:
- data-retention-policy-001 - ByteMage retains customer support data for a minimum of seven years un
- product-roadmap-001 - As of Q3 2026, ByteMage's top product priority is launching the AI Sea
- onboarding-guide-001 - New ByteMage employees complete orientation during their first week, i


## 10. Log Every Transformation

> For each request, store: original query, standalone query, query
> variations, sub-questions, selected retriever, retrieved chunk IDs. This
> supports evaluation and debugging later (Step 12).

No new machinery here — just call the functions from Sections 1-8 in order,
on one example, and collect what each step produced into a plain dict.

In [21]:
latest_message = "When did he do it?"

log = {"original_query": latest_message}

# Step 1: is this context-dependent?
if looks_context_dependent(latest_message):
    rewrite = rewrite_query(conversation_history, latest_message)
    log["standalone_query"] = rewrite.standalone_query
    log["needs_retrieval"] = rewrite.needs_retrieval
else:
    log["standalone_query"] = latest_message
    log["needs_retrieval"] = True

standalone_query = log["standalone_query"]

# Step 2: which route?
route, method = route_query(standalone_query)
log["selected_route"] = route
log["route_method"] = method

# Step 3: retrieve (expand into variations first, then fuse)
if log["needs_retrieval"] and route not in ("no_retrieval", "clarification_needed"):
    log["query_variations"] = generate_query_variations(standalone_query)
    results = multi_query_search(log["query_variations"])
    log["retrieved_chunk_ids"] = [r["chunk_id"] for r in results]
else:
    log["retrieved_chunk_ids"] = []

for key, value in log.items():
    print(f"{key}: {value}")

original_query: When did he do it?
standalone_query: When was ByteMage founded by Alan Whitfield and Priya Kapoor?
needs_retrieval: True
selected_route: database_lookup
route_method: model
query_variations: ['In what year was ByteMage established by Alan Whitfield and Priya Kapoor?', 'When did Alan Whitfield and Priya Kapoor start ByteMage?', 'What is the founding year of ByteMage founded by Alan Whitfield and Priya Kapoor?']
retrieved_chunk_ids: ['company-overview-001', 'onboarding-guide-001', 'data-retention-policy-001', 'leave-policy-001', 'compensation-policy-001']


### Reflection

- **Can rewriting change intent?** Yes — Section 3's example shows a naive
  rewrite can drop a scope word like "regular" vs "extended," which changes
  what's actually being asked.
- **Should the rewriter see the full conversation?** For a short conversation,
  yes. A very long one would need trimming or summarizing first.
- **How many query variations are useful?** Three is a reasonable default —
  enough to cover different wording without tripling retrieval cost.
- **When should a question be decomposed?** When it has genuinely separate
  parts (Section 6's leave example), not just because it's a long sentence.
- **Can HyDE introduce false assumptions?** Yes — the hypothetical passage is
  generated with no grounding, so if the model guesses wrong about the topic,
  the embedding steers retrieval the wrong way. Worth comparing against direct
  search, not trusting blindly.
- **Should the final answer see the rewritten query?** No — Section 2 is
  explicit that the rewrite is for retrieval only; generation should still
  answer the user's actual original message.
- **Can query expansion hurt precision?** Yes — more phrasings can pull in
  marginally-related chunks. Section 5's "found by more variations" rule helps
  because a chunk needs to show up repeatedly to rank highly.
- **When should the system ask for clarification instead of guessing?** When
  neither the conversation nor the message itself makes the request clear —
  that's what the router's `clarification_needed` route is for.